In [ ]:
import json
from collections import Counter, defaultdict
from pathlib import Path


def read_jsonl(path):
    """读取 JSONL 文件，逐行返回解析后的对象"""
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)


def normalize_counter_key(v):
    """将任意类型转为可计数的字符串（用于 Counter）"""
    if isinstance(v, (dict, list)):
        return json.dumps(v, ensure_ascii=False, sort_keys=True)
    return str(v)


def extract_value_evidence_fields(obj, path, counters):
    """
    递归遍历 obj，找出所有包含 value 和 evidence 键的字典。
    path: 当前路径（字符串，如 "result.setups"）
    counters: defaultdict(Counter)，用于累加统计
    """
    if isinstance(obj, dict):
        # 如果当前字典同时有 value 和 evidence，则视为一个 cell
        if "value" in obj and "evidence" in obj:
            # 移除冗余前缀，让报告更易读
            clean_key = path.replace("result.setups.", "").replace("result.", "")
            value = obj["value"]
            if value is not None:  # 增加空值过滤
                counters[clean_key][normalize_counter_key(value)] += 1
            return
        # 否则，继续遍历字典的每个键值对
        for k, v in obj.items():
            # 构建新路径：当前路径 + "." + k（如果 path 非空）
            new_path = f"{path}.{k}" if path else k
            extract_value_evidence_fields(v, new_path, counters)
    elif isinstance(obj, list):
        # 列表中的每个元素，路径不变（不引入索引）
        for item in obj:
            extract_value_evidence_fields(item, path, counters)
    # 其他类型忽略


def count_field_values(cleaning_data_path):
    """自动提取所有字段的 value 统计"""
    rows = list(read_jsonl(cleaning_data_path))
    counters = defaultdict(Counter)  # key: 字段路径, value: Counter
    for row in rows:
        # 从整个 JSON 对象开始，初始路径为空字符串
        extract_value_evidence_fields(row, "", counters)
    return counters


def get_topk(counter, top_n=20):
    """计算 top_n 的统计信息"""
    total_occurrences = sum(counter.values())
    unique_values_count = len(counter)
    res = []
    running_sum = 0
    for i, (value, count) in enumerate(counter.most_common(top_n), 1):
        running_sum += count
        ratio = (count / total_occurrences * 100) if total_occurrences else 0.0
        cumulative_ratio = (running_sum / total_occurrences * 100) if total_occurrences else 0.0
        res.append({
            "rank": i,
            "value": value,
            "count": count,
            "ratio": round(ratio, 4),
            "cumulative_ratio": round(cumulative_ratio, 4)
        })
    return {
        "total_occurrences": total_occurrences,
        "unique_values_count": unique_values_count,
        "top_k": res
    }


def show_field_report(domain, field_name, val_counter, top_n=20):
    """打印单个字段的报告"""
    total_occurrences = sum(val_counter.values())
    unique_values_count = len(val_counter)

    print("=" * 100)
    print(f"Domain: {domain}")
    print(f"Field : {field_name}")
    print(f"总抽取次数: {total_occurrences}")
    print(f"唯一值种类: {unique_values_count}")
    print("-" * 100)
    print(f"{'Rank':<8} | {'Value':<40} | {'Count':<10} | {'Ratio':>8} | {'Cumulative':>10}")
    print("-" * 100)

    running_sum = 0
    for i, (value, count) in enumerate(val_counter.most_common(top_n), 1):
        running_sum += count
        ratio = (count / total_occurrences * 100) if total_occurrences else 0.0
        cumulative_ratio = (running_sum / total_occurrences * 100) if total_occurrences else 0.0

        display_val = str(value)
        if len(display_val) > 37:
            display_val = display_val[:37] + "..."

        print(f"{i:<8} | {display_val:<40} | {count:<10} | {ratio:>7.2f}% | {cumulative_ratio:>9.2f}%")

    if total_occurrences > 0:
        covered = running_sum / total_occurrences * 100
    else:
        covered = 0.0

    print("-" * 100)
    print(f"结果：前 {top_n} 名累计覆盖了 {covered:.2f}% 的数据。")
    print()


def save_topk_json(output_path, domain, field_counters, top_n=20):
    """将所有字段的 top_k 结果保存为 JSON"""
    result = {
        "domain": domain,
        "top_k_by_field": {}
    }
    for field_name, counter in field_counters.items():
        result["top_k_by_field"][field_name] = get_topk(counter, top_n=top_n)

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)


def main(domain, top_n=20, print_report=True, save_json=True):
    cleaning_data_path = f"../../seed_data/{domain}/llm_extraction/raw_evidence_span.jsonl"
    output_json_path = f"../../seed_data/{domain}/llm_extraction/top{top_n}_by_field.json"

    # 自动提取所有字段的统计
    field_counters = count_field_values(cleaning_data_path)

    if print_report:
        for field_name, counter in field_counters.items():
            show_field_report(domain, field_name, counter, top_n=top_n)

    if save_json:
        save_topk_json(output_json_path, domain, field_counters, top_n=top_n)
        print(f"已保存到: {output_json_path}")


if __name__ == "__main__":
    domain =  "cell_culture"
    main(domain, top_n=10, print_report=True, save_json=False)

In [ ]:
import json
from collections import Counter, defaultdict
from pathlib import Path


def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)


def normalize_counter_key(v):
    """
    统一转成可计数的字符串。
    """
    if isinstance(v, (dict, list)):
        return json.dumps(v, ensure_ascii=False, sort_keys=True)
    return str(v)


def iter_cells(obj, parts):
    """
    递归找到某个 field_path 对应的所有 cell。
    例如:
      task
      electrolytes.chemical
      electrolytes.concentration
    """
    if isinstance(obj, list):
        for item in obj:
            yield from iter_cells(item, parts)
        return

    if not isinstance(obj, dict):
        return

    key = parts[0]
    if key not in obj:
        return

    if len(parts) == 1:
        cell = obj.get(key)
        if isinstance(cell, dict) and "value" in cell and "evidence" in cell:
            yield cell
        return

    yield from iter_cells(obj[key], parts[1:])


def count_field_values(cleaning_data_path, field_file_map_path):
    rows = list(read_jsonl(cleaning_data_path))
    field_map = read_json(field_file_map_path)
    field_paths = list(field_map.keys())

    field_counters = {field: Counter() for field in field_paths}

    for row in rows:
        setups = row.get("result", {}).get("setups", [])
        for setup in setups:
            for field_path in field_paths:
                for cell in iter_cells(setup, field_path.split(".")):
                    value = cell.get("value")
                    if value in (None, "", []):
                        continue

                    if isinstance(value, list):
                        for v in value:
                            field_counters[field_path][normalize_counter_key(v)] += 1
                    else:
                        field_counters[field_path][normalize_counter_key(value)] += 1

    return field_counters


def get_topk(counter, top_n=20):
    res = []
    total_occurrences = sum(counter.values())
    running_sum = 0

    for i, (value, count) in enumerate(counter.most_common(top_n), 1):
        running_sum += count
        ratio = (count / total_occurrences * 100) if total_occurrences else 0.0
        cumulative_ratio = (running_sum / total_occurrences * 100) if total_occurrences else 0.0

        res.append({
            "rank": i,
            "value": value,
            "count": count,
            "ratio": round(ratio, 4),
            "cumulative_ratio": round(cumulative_ratio, 4)
        })

    return {
        "total_occurrences": total_occurrences,
        "unique_values_count": len(counter),
        "top_k": res
    }


def show_field_report(domain, field_name, val_counter, top_n=20):
    total_occurrences = sum(val_counter.values())
    unique_values_count = len(val_counter)

    print("=" * 100)
    print(f"Domain: {domain}")
    print(f"Field : {field_name}")
    print(f"总抽取次数: {total_occurrences}")
    print(f"唯一值种类: {unique_values_count}")
    print("-" * 100)
    print(f"{'Rank':<8} | {'Value':<40} | {'Count':<10} | {'Ratio':>8} | {'Cumulative':>10}")
    print("-" * 100)

    running_sum = 0
    for i, (value, count) in enumerate(val_counter.most_common(top_n), 1):
        running_sum += count
        ratio = (count / total_occurrences * 100) if total_occurrences else 0.0
        cumulative_ratio = (running_sum / total_occurrences * 100) if total_occurrences else 0.0

        display_val = str(value)
        if len(display_val) > 37:
            display_val = display_val[:37] + "..."

        print(f"{i:<8} | {display_val:<40} | {count:<10} | {ratio:>7.2f}% | {cumulative_ratio:>9.2f}%")

    if total_occurrences > 0:
        covered = running_sum / total_occurrences * 100
    else:
        covered = 0.0

    print("-" * 100)
    print(f"结果：前 {top_n} 名累计覆盖了 {covered:.2f}% 的数据。")
    print()


def save_topk_json(output_path, domain, field_counters, top_n=20):
    result = {
        "domain": domain,
        "top_k_by_field": {}
    }

    for field_name, counter in field_counters.items():
        result["top_k_by_field"][field_name] = get_topk(counter, top_n=top_n)

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)


def main(domain, top_n=20, print_report=True, save_json=True):
    cleaning_data_path = f"../../seed_data/{domain}/open_llm_cleaning/evidence_span.jsonl"
    field_file_map_path = f"../../seed_data/{domain}/open_llm_extraction/field_file_map.json"
    output_json_path = f"../../seed_data/{domain}/open_llm_cleaning/top{top_n}_by_field.json"

    field_counters = count_field_values(cleaning_data_path, field_file_map_path)

    if print_report:
        for field_name, counter in field_counters.items():
            show_field_report(domain, field_name, counter, top_n=top_n)

    if save_json:
        save_topk_json(output_json_path, domain, field_counters, top_n=top_n)
        print(f"已保存到: {output_json_path}")


if __name__ == "__main__":
    domain = "cell_culture"
    main(domain, top_n=10, print_report=True, save_json=False)